In [ ]:


# ---------------- IMPORTS ----------------
import pandas as pd
import requests
import gradio as gr
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# ---------------- LOAD DATA ----------------
data = pd.read_csv("Crop_recommendation.csv")

X = data.drop("label", axis=1)
y = data["label"]

crop_model = RandomForestClassifier()
crop_model.fit(X, y)

# ---------------- YIELD MODEL ----------------
data["yield_score"] = (
    data["temperature"] * 0.3 +
    data["humidity"] * 0.2 +
    data["rainfall"] * 0.5
)

X_yield = data.drop(["label", "yield_score"], axis=1)
y_yield = data["yield_score"]

yield_model = RandomForestRegressor()
yield_model.fit(X_yield, y_yield)

# ---------------- API ----------------
API_KEY = "aa08b5c4a3d6b872cbf894fdacdf089a"  # Replace with your OpenWeatherMap API key

def get_weather(city):
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"
    res = requests.get(url).json()

    if "main" not in res:
        raise Exception(res.get("message", "Invalid city or API error"))

    temp = res["main"]["temp"]
    humidity = res["main"]["humidity"]
    rainfall = res.get("rain", {}).get("1h", 5) * 10

    return temp, humidity, rainfall

# ---------------- SOIL ----------------
def get_soil():
    sample = data.sample(1)
    return (
        sample["N"].values[0],
        sample["P"].values[0],
        sample["K"].values[0],
        sample["ph"].values[0]
    )

# ---------------- SOIL HEALTH ----------------
def get_soil_health(N, P, K):
    status = []

    status.append("🌿 Nitrogen: High" if N > 100 else "🌿 Nitrogen: Low")
    status.append("🌸 Phosphorus: High" if P > 50 else "🌸 Phosphorus: Low")
    status.append("💪 Potassium: High" if K > 50 else "💪 Potassium: Low")

    return "\n".join(status)

# ---------------- GRAPH FUNCTIONS ----------------
def create_soil_chart(N, P, K):
    fig, ax = plt.subplots()
    labels = ["Nitrogen (N)", "Phosphorus (P)", "Potassium (K)"]
    values = [N, P, K]

    ax.bar(labels, values)
    ax.set_title("Soil Nutrient Levels")
    ax.set_ylabel("Concentration")

    return fig

def create_weather_chart(temp, humidity, rainfall):
    fig, ax = plt.subplots()
    ax.bar(["Temperature", "Humidity", "Rainfall"], [temp, humidity, rainfall])
    ax.set_title("Weather Conditions")
    return fig

def create_yield_chart(yield_value):
    fig, ax = plt.subplots()
    ax.bar(["Yield Score"], [yield_value])
    ax.set_title("Predicted Yield")
    return fig

def create_weather_trend(temp, humidity, rainfall):
    fig, ax = plt.subplots()

    temps = [temp-2, temp-1, temp, temp+1, temp+2]
    humidity_vals = [humidity-5, humidity-2, humidity, humidity+2, humidity+5]
    rainfall_vals = [rainfall-10, rainfall-5, rainfall, rainfall+5, rainfall+10]

    ax.plot(temps, label="Temperature")
    ax.plot(humidity_vals, label="Humidity")
    ax.plot(rainfall_vals, label="Rainfall")

    ax.set_title("Weather Trend (Simulated)")
    ax.legend()

    return fig

def create_soil_pie(N, P, K):
    fig, ax = plt.subplots()
    labels = ["Nitrogen", "Phosphorus", "Potassium"]
    values = [N, P, K]

    ax.pie(values, labels=labels, autopct='%1.1f%%')
    ax.set_title("Soil Composition")

    return fig

def create_comparison_chart(temp, humidity, rainfall):
    fig, ax = plt.subplots()

    labels = ["Temperature", "Humidity", "Rainfall"]
    current = [temp, humidity, rainfall]
    ideal = [25, 60, 100]

    x = range(len(labels))

    ax.plot(x, current, marker='o', label="Current")
    ax.plot(x, ideal, marker='o', label="Ideal")

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_title("Current vs Ideal Conditions")
    ax.legend()

    return fig

# ---------------- MAIN FUNCTION ----------------
def predict(city):
    try:
        temp, humidity, rainfall = get_weather(city)
        N, P, K, ph = get_soil()

        input_data = pd.DataFrame([{
            "N": N,
            "P": P,
            "K": K,
            "temperature": temp,
            "humidity": humidity,
            "ph": ph,
            "rainfall": rainfall
        }])

        crop = crop_model.predict(input_data)[0]
        yield_value = yield_model.predict(input_data)[0]

        soil_health = get_soil_health(N, P, K)

        # Graphs
        soil_fig = create_soil_chart(N, P, K)
        weather_fig = create_weather_chart(temp, humidity, rainfall)
        yield_fig = create_yield_chart(yield_value)
        trend_fig = create_weather_trend(temp, humidity, rainfall)
        pie_fig = create_soil_pie(N, P, K)
        compare_fig = create_comparison_chart(temp, humidity, rainfall)

        result_text = f"""
# 🌾 AI Crop Recommendation

## 🌱 Recommended Crop
*{crop}*

---

## 🌿 Soil Health
{soil_health}

---

## 📊 Details
- Yield Score: {round(yield_value, 2)}
- Temperature: {temp} °C
- Humidity: {humidity} %
- Rainfall: {rainfall} mm
"""

        return (
            result_text,
            soil_fig,
            weather_fig,
            yield_fig,
            trend_fig,
            pie_fig,
            compare_fig
        )

    except Exception as e:
        return f"❌ Error: {str(e)}", None, None, None, None, None, None

# ---------------- UI ----------------
with gr.Blocks() as app:
    gr.Markdown("# 🌾 AI Crop Recommendation System")

    city_input = gr.Textbox(label="Enter Region", value="Bangalore")
    submit_btn = gr.Button("Predict")

    output_text = gr.Markdown()

    gr.Markdown("## 📊 Visual Insights")

    soil_plot = gr.Plot(label="Soil Nutrients")
    weather_plot = gr.Plot(label="Weather Data")
    yield_plot = gr.Plot(label="Yield Prediction")

    trend_plot = gr.Plot(label="Weather Trend")
    pie_plot = gr.Plot(label="Soil Composition")
    compare_plot = gr.Plot(label="Current vs Ideal")

    submit_btn.click(
        fn=predict,
        inputs=city_input,
        outputs=[
            output_text,
            soil_plot,
            weather_plot,
            yield_plot,
            trend_plot,
            pie_plot,
            compare_plot
        ]
    )

app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a04a9be940f6920459.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
!pip install gradio scikit-learn matplotlib pandas requests


In [16]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 